# ✨ Nexra.ai — AI Tutor on Google Colab

Runs the FastAPI tutor backend on a free Colab GPU and gives you a **public link** with the chat UI.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU` (or better).
2. The default model `mistralai/Mistral-7B-Instruct-v0.2` is *gated*: accept its terms on
   [huggingface.co](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2), create a token at
   https://huggingface.co/settings/tokens, then add it in Colab's 🔑 **Secrets** panel as `HF_TOKEN`
   (enable "Notebook access").
3. Run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code
Clones the GitHub repo. If the repo is private or you have local edits, instead upload
`main.py`, `index.html` and `requirements.txt` with the 📁 Files panel into `/content/Local-tune-LLM`.

In [ ]:
import os
REPO_URL = "https://github.com/ankitbisht020/Local-tune-LLM.git"
APP_DIR = "/content/Local-tune-LLM"

if not os.path.exists(os.path.join(APP_DIR, "main.py")):
    !git clone -q {REPO_URL} {APP_DIR} || mkdir -p {APP_DIR}
if not os.path.exists(os.path.join(APP_DIR, "main.py")):
    from google.colab import files
    print("Clone failed - please upload main.py, index.html and requirements.txt")
    os.chdir(APP_DIR)
    files.upload()
os.chdir(APP_DIR)
!ls -la

## 3. Install dependencies (Colab already has PyTorch)

In [ ]:
!pip install -q -r requirements.txt

## 4. Settings
Change `MODEL_ID` to try another model. Good options that fit a free T4 in 4-bit:
- `mistralai/Mistral-7B-Instruct-v0.2` (default, gated)
- `mistralai/Mistral-7B-Instruct-v0.3` (gated)
- `Qwen/Qwen2.5-7B-Instruct` (not gated, very strong at maths/science)
- `meta-llama/Llama-3.1-8B-Instruct` (gated)

In [ ]:
import os
os.environ["MODEL_ID"] = "mistralai/Mistral-7B-Instruct-v0.2"
os.environ["MAX_NEW_TOKENS"] = "1024"
os.environ["TEMPERATURE"] = "0.3"
os.environ["PORT"] = "8001"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception as e:
    print("No HF_TOKEN secret found (only needed for gated models):", e)

## 5. Start the server (first run downloads ~15 GB and takes a few minutes)

In [ ]:
import subprocess, time, requests, os

PORT = int(os.environ["PORT"])
if "server" in globals() and server.poll() is None:
    server.terminate(); server.wait()

log = open("/content/server.log", "w")
server = subprocess.Popen(["python", "main.py"], cwd=APP_DIR, stdout=log, stderr=subprocess.STDOUT)

start = time.time()
while True:
    time.sleep(5)
    if server.poll() is not None:
        print(open("/content/server.log").read()[-4000:])
        raise RuntimeError("Server exited - see log above")
    try:
        h = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3).json()
        if h.get("ready"):
            print(f"✅ Ready in {time.time()-start:.0f}s:", h); break
        if h.get("status") == "error":
            print(open("/content/server.log").read()[-4000:]); raise RuntimeError(h.get("error"))
    except requests.exceptions.ConnectionError:
        pass
    lines = open("/content/server.log").read().strip().splitlines()
    print(f"⏳ loading... {time.time()-start:.0f}s | " + (lines[-1][-120:] if lines else "starting"))

## 6. Get a public link (Cloudflare quick tunnel — no account needed)
Open the printed `https://….trycloudflare.com` link in any browser or phone to use the tutor.

In [ ]:
import re, subprocess, time
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared

if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate(); tunnel.wait()
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        public_url = m.group(0); break
print("🌐 Open the tutor:", public_url or "not found - check /content/tunnel.log")
print("📘 API docs:      ", f"{public_url}/docs" if public_url else "")

## 7. Quick test (streams an answer right here)

In [ ]:
import json, requests
payload = {"userId": "colab-test", "studentClass": 8, "text": "What is photosynthesis?"}
with requests.post(f"http://127.0.0.1:{PORT}/ask/stream", json=payload, stream=True) as r:
    for line in r.iter_lines(decode_unicode=True):
        if line and line.startswith("data:"):
            event = json.loads(line[5:])
            if "token" in event: print(event["token"], end="", flush=True)
            elif "done" in event: print(f"\n\n--- {event['seconds']}s, stop: {event['stopReason']}")
            elif "error" in event: print("ERROR:", event["error"])

## Logs / stop

In [ ]:
!tail -n 30 /content/server.log

In [ ]:
# Run to shut everything down
tunnel.terminate(); server.terminate()